# Stockout EDA

Exploratory analysis for top-15 SKU stockout prediction. The modeling target is daily binary stockout at store-SKU level: `stock_hour6_22_cnt > 0`.

In [ ]:
# Colab bootstrap: clone or update the repo, then run from its root.
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/vibhor-5/btp.git"
REPO_DIR = Path("/content/btp")
try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    os.chdir(REPO_DIR)
else:
    os.chdir(Path.cwd())

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
TRAIN_PATH = DATA_DIR / "top15_train.parquet"
TEST_PATH = DATA_DIR / "top15_test.parquet"
print("Project root:", PROJECT_ROOT)
print("Train exists:", TRAIN_PATH.exists(), TRAIN_PATH)
print("Test exists:", TEST_PATH.exists(), TEST_PATH)

try:
    import pyarrow  # noqa: F401
except ImportError:
    !pip -q install pyarrow

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

In [ ]:
train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

for name, df in [("train", train_df), ("test", test_df)]:
    df["dt"] = pd.to_datetime(df["dt"])
    df["stockout"] = (df["stock_hour6_22_cnt"] > 0).astype(int)
    print(f"\n{name}: shape={df.shape}")
    print("date range:", df["dt"].min(), "to", df["dt"].max(), "unique days:", df["dt"].nunique())
    print("stores:", df["store_id"].nunique(), "products:", df["product_id"].nunique(), "store-product groups:", df.groupby(["store_id", "product_id"]).ngroups)
    print("stockout rate:", df["stockout"].mean().round(4), "positive rows:", int(df["stockout"].sum()))

display(train_df.head())
display(train_df.dtypes.to_frame("dtype"))

In [ ]:
combined = pd.concat([train_df.assign(split="train"), test_df.assign(split="test")], ignore_index=True)

missing = combined.isna().mean().sort_values(ascending=False).to_frame("missing_rate")
display(missing[missing["missing_rate"] > 0])

target_summary = combined.groupby("split").agg(
    rows=("stockout", "size"),
    stockout_rate=("stockout", "mean"),
    stockout_hours_mean=("stock_hour6_22_cnt", "mean"),
    stockout_hours_p95=("stock_hour6_22_cnt", lambda x: np.percentile(x, 95)),
)
display(target_summary)

In [ ]:
daily = combined.groupby(["split", "dt"], as_index=False).agg(
    stockout_rate=("stockout", "mean"),
    rows=("stockout", "size"),
    sale_amount_mean=("sale_amount", "mean"),
)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
sns.lineplot(data=daily, x="dt", y="stockout_rate", hue="split", marker="o", ax=axes[0])
axes[0].set_title("Daily stockout rate")
sns.lineplot(data=daily, x="dt", y="sale_amount_mean", hue="split", marker="o", ax=axes[1])
axes[1].set_title("Daily mean sale amount")
plt.tight_layout()

In [ ]:
sku_summary = combined.groupby("product_id", as_index=False).agg(
    rows=("stockout", "size"),
    stockout_rate=("stockout", "mean"),
    sale_mean=("sale_amount", "mean"),
    sale_std=("sale_amount", "std"),
    stock_hours_mean=("stock_hour6_22_cnt", "mean"),
).sort_values("stockout_rate", ascending=False)
display(sku_summary)

plt.figure(figsize=(12, 5))
sns.barplot(data=sku_summary, x="product_id", y="stockout_rate", order=sku_summary["product_id"].astype(str))
plt.title("Stockout rate by product")
plt.xticks(rotation=45)
plt.tight_layout()

In [ ]:
store_summary = combined.groupby("store_id", as_index=False).agg(
    rows=("stockout", "size"),
    stockout_rate=("stockout", "mean"),
).sort_values("stockout_rate", ascending=False)

display(store_summary.head(20))

plt.figure(figsize=(12, 5))
sns.histplot(store_summary["stockout_rate"], bins=40)
plt.title("Distribution of store-level stockout rates")
plt.tight_layout()

In [ ]:
num_cols = ["sale_amount", "discount", "precpt", "avg_temperature", "avg_humidity", "avg_wind_level", "stock_hour6_22_cnt"]
available_num_cols = [c for c in num_cols if c in combined.columns]

display(combined[available_num_cols + ["stockout"]].describe().T)

sample = combined.sample(min(len(combined), 50000), random_state=42)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), [c for c in available_num_cols if c != "stock_hour6_22_cnt"][:6]):
    sns.boxplot(data=sample, x="stockout", y=col, ax=ax, showfliers=False)
    ax.set_title(col)
plt.tight_layout()

In [ ]:
group_sizes = combined.groupby(["store_id", "product_id"]).size()
display(group_sizes.describe().to_frame("rows_per_store_product"))

expected_days = combined["dt"].nunique()
continuity = combined.groupby(["store_id", "product_id"])["dt"].nunique().rename("unique_days").reset_index()
display(continuity["unique_days"].describe().to_frame())
print("Groups with full coverage:", int((continuity["unique_days"] == expected_days).sum()), "/", len(continuity))